In [ ]:
!pip install -q gradio

## 文字輸入框 & 輸出框

In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch(share=True) # share=True 會產生公開網址



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2ecbcec3b272ce2b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 輸入框 Label 與 Placeholder

In [ ]:
import gradio as gr

def greet(name):
    return "你好 " + name

# 實例化輸入組件
input_textbox = gr.Textbox(
    label="輸入你的名稱：",
    placeholder="Your name",
    lines=2
)

# 實例化輸出組件 (建議也定義 label，介面更美觀)
output_textbox = gr.Textbox(label="系統回覆")

gr.Interface(
    fn=greet,
    inputs=input_textbox,
    outputs=output_textbox
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4981eb1e6fa2f8bbf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 範例1：隨機回應「Yes」或「No」

In [ ]:
import random
import gradio as gr

# 定義處理函式，必須包含 message 與 history 兩個參數
# ChatInterface 會自動將之前的對話紀錄傳入 history 參數
def random_response(message, history):
    # 1. 檢查 history 是否為空
    if not history:
        print("--- 這是第一輪對話，所以 history 是空的 ---")
    else:
        print(f"--- 當前歷史紀錄層級 (長度: {len(history)}) ---")
        print(history)

    # 2. 顯示當前收到的訊息
    print(f"目前使用者輸入: {message}")

    return random.choice(["Yes", "No"])

# 使用 ChatInterface 綁定函式
gr.ChatInterface(
    fn=random_response,
    type="messages" # 建議設定，將對話格式化為訊息清單
).launch(share=True, debug=True)  # 有設定 debug 模式，rint 才會印出來

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bd61d8bf75e877f0c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- 這是第一輪對話，所以 history 是空的 ---
目前使用者輸入: 1
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7866 <> https://4e8bdea0f21ab66b66.gradio.live
Killing tunnel 127.0.0.1:7867 <> https://0bbfe25f0d4d1b00e8.gradio.live
Killing tunnel 127.0.0.1:7868 <> https://bd61d8bf75e877f0c7.gradio.live


## 範例2：依序「同意」或「不同意」的簡單邏輯

In [ ]:
import gradio as gr

def alternatingly_agree(message, history):
    # 計算助理出現的次數，來判斷要「同意」或「不同意」
    # 這裡過濾出歷史紀錄中角色為 assistant 的訊息
    assistant_count = len([h for h in history if h['role'] == "assistant"])

    if assistant_count % 2 == 0:
        return f"Yes, I do think that: {message}"
    else:
        return "I don't think so"

# 建立 Gradio 聊天介面
gr.ChatInterface(
    fn=alternatingly_agree,
    type="messages"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://305af8bced5691f39d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 範例3：串流(Streaming)聊天機器人

這段程式碼展示了如何在 Gradio 中實現 **「串流 (Streaming)」** 效果。這是現代 AI 聊天機器人（如 ChatGPT）必備的功能，能讓使用者在模型生成長篇大論時，即時看到已經產出的文字，而不是空等好幾秒後才一次噴出所有結果。

### 程式碼核心邏輯解釋

1. 關鍵關鍵字：`yield`
* 在 Python 中，`yield` 會將函式轉化為一個**生成器 (Generator)**。
* 與 `return` 不同（`return` 執行後會直接結束函式並回傳最終結果），`yield` 可以**多次回傳**內容。
* 在 Gradio 的 `ChatInterface` 中，每當 `yield` 送出一個新的字串，前端介面就會即時更新該則訊息的內容。


2. 模擬打字效果：`time.sleep(0.3)`
* 這行程式碼刻意讓程式每跑一個迴圈就停頓 0.3 秒，模擬人類打字或是 AI 運算時的延遲感。


3. 切片語法：`message[: i+1]`
* 這是一個 Python 的字串切片（Slicing）。
* 假設 `message` 是 "Hello"：
* 第一次迴圈：`yield "You typed: H"`
* 第二次迴圈：`yield "You typed: He"`
* 以此類推，直到顯示完整訊息。


* **注意：** 每次 `yield` 的內容都會**覆蓋掉**該則訊息之前的內容，所以我們必須每次都回傳從開頭到當前字元的所有文字。

### 真實應用場景的思考

* **使用者體驗 (UX)**：在電商客服場景中，若 API 回應較慢，串流能有效降低使用者的焦慮感，讓他們知道系統「正在處理中」。
* **中斷機制**：如圖中說明提到，當進入串流狀態時，Gradio 的按鈕會自動變成 「Stop」，這讓使用者有權利在不滿意回答時隨時喊停，節省 API Token 的消耗。
* **邏輯驗證**：請注意，這段代碼的 `history` 參數雖然存在但並未被使用。如果你要結合上一段的「同意/不同意」邏輯，你需要先計算 `history` 的次數，再進入 `for` 迴圈進行 `yield`。


In [ ]:
import time
import gradio as gr

def slow_echo(message, history):
    # 根據輸入訊息的長度進行迴圈
    for i in range(len(message)):
        # 暫停 0.3 秒，模擬打字過程
        time.sleep(0.3)

        # 使用 yield 逐字回傳目前已累積的字串內容
        # 語法 message[: i+1] 代表從索引 0 取到 i
        yield "You typed: " + message[: i+1]

# 建立 Gradio 聊天介面
gr.ChatInterface(
    fn=slow_echo,
    type="messages"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5807f7f8cdbd87b7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 範例4：額外的輸入(Additional Inputs)

在實際應用中，除了對話內容，我們常需要調整後台參數（如系統提示詞或回應長度限制）。這段代碼示範了如何利用 `additional_inputs` 增加控制欄位。

* **多參數處理**：`echo` 函式除了接收 `message` 和 `history`，還接收了 `system_prompt` 和 `tokens`。
* **UI 組件整合**：
* `gr.Textbox`：用於輸入系統指令（System Prompt），這在企業情境中定義角色設定（例如：你是一名客服人員）非常有用。
* `gr.Slider`：用於控制回應長度（tokens），限制生成字數。


* **動態限制**：在第 6 行中使用 `min(len(response), int(tokens))`，確保輸出的長度不會超過拉桿所設定的上限。

### 商業啟發與建議

這兩個功能結合起來後，你就具備了開發**專業級 AI 客服介面**的基本工具。你可以設定一個不可見或隱藏的 `System Prompt` 來約束 AI 的語氣，並提供串流回傳讓使用者覺得系統反應迅速。

如果你想更進一步，我可以教你如何將這些 UI 元件與真正的 **OpenAI** 或 **Anthropic** API 對接，讓它從「模擬打字」變成真正的「AI 對話」，需要我試寫一個版本給你參考嗎？

### 參數映射規則

`gr.ChatInterface` 的設計邏輯非常直覺且具備固定順序。

當你在 `additional_inputs` 中定義了額外的輸入元件時，Gradio 會依照**列表中的順序**，自動將這些元件的值映射到處理函數（fn）中**緊接在 `message` 與 `history` 之後**的參數。

處理函數的參數結構如下：

1. **第一個參數**：使用者輸入的訊息 (`message`)。
2. **第二個參數**：對話歷史紀錄 (`history`)。
3. **後續參數**：對應到 `additional_inputs` 列表中的元件值。

### 注意事項與建議

* **名稱無關，順序至上**：處理函數中的參數名稱（例如你取名叫 `tokens` 或 `limit`）並不重要，重要的是它們在參數列表中的**物理位置**。
* **預設值處理**：如果在 `additional_inputs` 的元件中設定了 `value`，該預設值會在第一次呼叫時自動傳入對應的參數。
* **擴充性**：如果你在 `additional_inputs` 增加了第三個元件（例如一個勾選框 `gr.Checkbox`），你就必須在 `echo` 函數中增加第五個參數來接收它。

這種自動映射機制簡化了開發流程，讓你不需要手動去連接 UI 元件與後端邏輯。


In [ ]:
import gradio as gr
import time

# system_prompt -> 對應 additional_inputs[0]
# tokens -> 對應 additional_inputs[1]
def echo(message, history, system_prompt, tokens):
    # 結合系統提示詞與使用者訊息
    response = f"System prompt: {system_prompt}\nMessage: {message}"

    # 根據 tokens 限制決定輸出的長度與串流效果
    for i in range(min(len(response), int(tokens))):
        time.sleep(0.05)
        yield response[: i + 1]

demo = gr.ChatInterface(
    echo,
    type="messages",
    additional_inputs=[
        # 設定預設文字與標籤 -- 第一個額外輸入 -> 對應第三個參數
        gr.Textbox("You are helpful AI.", label="System Prompt"),
        # 設定滑桿範圍 10 到 100 -- 第二個額外輸入 -> 對應第四個參數
        gr.Slider(10, 100),
    ],
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70cd86319b157c1187.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 範例5：預設回覆(Preset Responses)

這段程式碼展示了如何使用 Python 的 **Gradio** 庫（特別是 `gr.ChatInterface`）來建立一個具有預設回覆按鈕（Options）聊天介面。這在需要引導使用者點擊特定選項而非手動輸入時非常有用。

### 程式碼核心邏輯解釋

1. **資料結構轉向**：
在第 27 行設置了 `type="messages"`。這意味著 `chat` 函式不再只是回傳簡單的字串，而是回傳一個包含 `role`、`content` 與 `options` 的**字典（Dictionary）**。
2. **選項按鈕（Options）**：
第 19-22 行定義了兩個按鈕：
* **Yes**：當使用者點擊時，會傳送 `value` 中的內容（"Yes, that's correct."）給後端。
* **No**：點擊後傳送 "No"。


3. **條件判斷**：
第 13-14 行處理使用者的回饋。如果使用者點擊了 "Yes" 按鈕，系統就會辨識到特定的字串並回傳 "Great!"。

### 延伸思考：真的「簡化」了流程嗎？

雖然 `options` 提供了便利性，但從使用者經驗（UX）的角度來看，這也限制了對話的開放性。

* **優點**：降低輸入成本，適合表單式對話或結構化回饋。
* **潛在問題**：如果 `options` 沒設計好，使用者可能會在「想說的話」與「按鈕上的話」之間感到矛盾。



In [ ]:
import gradio as gr

# 預先定義要顯示的範例程式碼字串
example_code = '''
Here's the code I generated:

def greet(x):
    return f"Hello, {x}!"

Is this correct?
'''

def chat(message, history):
    # 如果使用者點擊了預設按鈕 "Yes"
    if message == "Yes, that's correct.":
        return "Great!"
    else:
        # 第一次回覆或非特定訊息時，回傳帶有選項的字典
        return {
            "role": "assistant",
            "content": example_code,
            "options": [
                {"value": "Yes, that's correct.", "label": "Yes"},
                {"value": "No"}
            ]
        }

# 使用 ChatInterface 建立介面
demo = gr.ChatInterface(
    chat,
    type="messages", # 關鍵設定：允許回傳字典格式以支援選項
    examples=["Write a Python function that takes a string and returns a greeting."]
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ab81618acfcad5de0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 練習題

此練習題主題為如何為 Chatbot 加入**系統提示（System Prompt）**與**字數限制**的額外輸入。

## 為 Chatbot 加入「系統提示」與「字數限制」的額外輸入

我們先用幾個小範例來示範如何接收訊息、產出回覆、使用者介面呈現，以及串流或多種附加輸入等更多可能性。接下來，你可以嘗試將這些技術組合起來，根據實際專案需求客製化功能，或結合更強大的 AI 模型進行語言處理。

### ● 題目描述

請嘗試在原有的 Chatbot 基礎上，新增兩個額外輸入：

1. **一個 Textbox** 作為「系統提示」（System Prompt），此文字內容會影響整體對話風格或角色設定。
2. **一個 Slider** 用來控制 Chatbot 回應的最大字數（或 Token 數）。

建立此多輸入 Chatbot 時，請參考上方材料或自行翻閱 Gradio 文件中的 `additional_inputs` 章節，並設計一個函式能夠在產生回應時依照使用者的「系統提示」與「字數限制」執行。

### ● 情境

想像你在開發一個專業諮詢聊天機器人，你需要在每次對話前指定諮詢領域或角色（例如：心理諮商師、理財顧問、健身教練），同時希望能在系統內控管回應長度，避免過度冗長或不符合風格。這時就可以利用「額外輸入」的概念為 Chatbot 添加更多動態參數。

### ● 完整要求

1. **建立一個具備兩個額外輸入的 Chatbot：**

  * **Textbox**：擔任「系統提示」，可以讓使用者修改角色設定。
  * **Slider**：控制最大回應字數（或 Token 數）。

2. 在回應函式中，需同時讀取**文字輸入**、**系統提示**，以及**對話紀錄（history）**。
3. 以**串流（yield）**方式回傳 Chatbot 的回答，確保每次輸出的累積不超過 `max_length`。
4. 在界面上提供**標題與描述**，協助使用者理解該 Chatbot 的用途。


### 實作

### 1. 核心處理函式：`role_based_responder`

這是 Chatbot 的大腦，處理輸入並生成輸出：

* **參數：** 接收使用者訊息 (`message`)、歷史紀錄 (`history`)，以及兩個自定義輸入：系統提示詞 (`role_prompt`) 與最大字數 (`max_length`)。
* **提示詞整合：** 將角色設定與使用者訊息組合成 `combined_prompt`，這在實際開發中是發送給 AI 模型的內容。
* **回應模擬：** 這裡使用 F-string 產生一個假設性的回應，模擬 AI 已經理解了角色。
* **串流效果 (`yield`)：** 使用 `[:int(max_length)]` 根據滑桿數值截斷回應。
* 透過 `for` 迴圈搭配 `time.sleep(0.05)`，讓文字像打字機一樣逐一出現，提升使用者體驗。

### 2. 使用者介面：`gr.ChatInterface`

這是 Gradio 提供的高階組件，專門用於快速搭建聊天窗口：

* **`fn`**：指定處理訊息的函式。
* **`additional_inputs`**：這是一個關鍵點。除了標準的輸入框，它額外增加了：
* `gr.Textbox`：讓使用者隨時更改 AI 的角色（例如：心理諮商師、英文老師）。
* `gr.Slider`：讓使用者動態調整 AI 回應的長度限制。
* **`demo.launch()`**：啟動網頁伺服器，讓你在瀏覽器中操作介面。


In [ ]:
import gradio as gr
import time

def role_based_responder(message, history, role_prompt, max_length):
    """
    - message: 使用者輸入的文字
    - history: 過往對話紀錄
    - role_prompt: 系統提示，決定 Chatbot 的角色或風格
    - max_length: 回應的最大字數
    """

    # 將系統提示與使用者訊息整合，形成 Chatbot 回應的基礎
    combined_prompt = f"【系統提示】{role_prompt}\n【使用者訊息】{message}\n"

    # 建立一個假設的回應（在真實情況中可串接 LLM API）
    full_response = f"你好！我是你設定的角色：{role_prompt}\n關於你的問題：{message}\n以下是我的建議..."

    # 使用 yield 做串流回應，並根據 max_length 控制輸出長度
    truncated_response = full_response[:int(max_length)]

    for i in range(len(truncated_response)):
        time.sleep(0.05)  # 模擬模型思考時間
        yield truncated_response[:i+1]

demo = gr.ChatInterface(
    fn=role_based_responder,
    type="messages",
    additional_inputs=[
        gr.Textbox(
            value="請扮演一位溫暖且耐心的心理諮商師",
            label="系統提示 (Role Prompt)"
        ),
        gr.Slider(10, 200, step=10, value=100, label="回應字數上限 (Max Length)")
    ],
    title="角色扮演與字數控制 Chatbot",
    description="依照系統提示扮演不同角色，並用滑桿控制回應字數長度。"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2314aae396df0ac4dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
